# BSISO1 phase composites: Ekman transport, MSE and Ekman advection

Composites by BSISO1 phase (active days, amplitude > 1) from `bsiso_phase.jl`, file `ekman_bsiso1_phase_2012-2020.nc`. The first figures show one phase `p`, as in `ekman_plots.ipynb` for a pentad; the last two show zonal wind stress and surface-air MSE for all 8 phases, to pick out the phases of interest (e.g. monsoon onset). The library `ekman_V_ma_mo_plot.jl` is loaded with Revise `includet`.

Kernel: "Julia", started with `--project=@.`, which finds `plot/Project.toml`.

In [ ]:
using Revise
includet("ekman_V_ma_mo_plot.jl")

## Data
Phase `p` in the region `lonlim`, `latlim`; only that region is read. `d`: composite fields as read; `f`: the fields plotted. `ndays`: the number of active days composited in a phase (the maximum of `nobs_taux` over the region). `Ro_rad` curtails M near the equator after compositing (see `ekman_fields`).

In [ ]:
file = "../ekman_bsiso1_phase_2012-2020.nc"
p = 4                                # BSISO1 phase (1-8)
lonlim, latlim = (40, 110), (-5, 30) # degrees: Indian Ocean; only this region is read
Ro_rad = 250e3 # m, curtail M by r = 1 - exp(-y^2/Ro_rad^2) near the equator (plotting only)
advlim = 180   # W m^-2, color limit for the enthalpy advection panels
phases = NCDataset(nc -> nc["phase"][:], file) # 1:8
ndays(k) = Int(maximum(read_group(file, k; vars=("nobs_taux",), lonlim, latlim).nobs_taux))
k = findfirst(==(p), phases)
d = read_group(file, k; lonlim, latlim)
f = ekman_fields(d; Ro_rad)
figtitle = "$(basename(file)): BSISO1 phase $p, $(ndays(k)) days (Ro_rad = $(Ro_rad/1e3) km)"
size(f.Mx), keys(f)

## Ekman transport, surface-air MSE and ocean-surface enthalpy, phase `p`
Each figure is a list of `panel(x, title, units; cmap="RdBu_r", kw...)`, filled in row by row (2 columns). `kw` go to `pcolormesh`; without vmin/vmax the limits are ±98th percentile of |x|. The MSE limits are copied from the pentad notebook; adjust them for the phase.

In [ ]:
panels1 = [
    panel(f.Mx, raw"Ekman transport $M_x$ (ocean)", raw"kg m$^{-1}$ s$^{-1}$"),
    panel(f.My, raw"Ekman transport $M_y$ (ocean)", raw"kg m$^{-1}$ s$^{-1}$"),
    panel(f.m_o ./ 1e3, raw"ocean surface enthalpy $m_o = c_{po}$ SST", raw"kJ kg$^{-1}$"; cmap="viridis", vmin=1195, vmax=1215),
    panel(f.m_a ./ 1e3, raw"surface air MSE $m_a = c_{pa} T_2 + L_v q$", raw"kJ kg$^{-1}$"; cmap="viridis", vmin=345, vmax=365),
    panel(f.adv_o, raw"ocean Ekman advection $-M_o \cdot \nabla m_o$", flux_units; vmin=-advlim, vmax=advlim),
    panel(f.adv_a, raw"atmosphere Ekman advection $-M_a \cdot \nabla m_a$, $M_a = -M_o$", flux_units; vmin=-advlim, vmax=advlim)
    ]    
fig1, axs1 = plot_panels(f.lon, f.lat, panels1; land=f.land, suptitle=figtitle);

## Ocean and atmosphere Ekman advection and their sum, phase `p`
Columns: ocean, atmosphere, sum. Rows: temperature (mass-weighted, K kg m$^{-2}$ s$^{-1}$) and (moist) enthalpy, W m$^{-2}$. Each row shares its color limits.

In [ ]:
Tlim = maximum(symlim, (f.advT_o, f.advT_a)) # shared by the temperature row
Hlim = advlim                                # W m^-2, shared by the enthalpy row
panels_adv = [
    panel(f.advT_o, raw"ocean advection $-M_o \cdot \nabla$SST", advT_units; vmin=-Tlim, vmax=Tlim),
    panel(f.advT_a, raw"atmosphere advection $-M_a \cdot \nabla T_2$", advT_units; vmin=-Tlim, vmax=Tlim),
    panel(f.advT_o .+ f.advT_a, "oc+atm T advection sum", advT_units; vmin=-Tlim, vmax=Tlim),
    panel(f.adv_o, raw"ocean enthalpy advection $-M_o \cdot \nabla m_o$, $m_o = c_{po}$SST", flux_units; vmin=-Hlim, vmax=Hlim),
    panel(f.adv_a, raw"atmosphere moist enthalpy advection $-M_a \cdot \nabla m_a$, $m_a = c_{pa}T_2 + L_v q$", flux_units; vmin=-Hlim, vmax=Hlim),
    panel(f.adv_o .+ f.adv_a, "oc+atm advection sum", flux_units; vmin=-Hlim, vmax=Hlim)]
fig_adv, axs_adv = plot_panels(f.lon, f.lat, panels_adv; ncols=3, land=f.land, suptitle=figtitle);

## Cancellation of Ekman divergence and flux divergence tendency, phase `p`
The flux form $-\nabla\cdot(M m)$ is from the saved composites of $s\tau_x$, $s\tau_y$; $m \nabla\cdot M = $ adv $-$ flux.

In [ ]:
panels3 = [
    panel(f.sdiv_o, raw"ocean $m_o \nabla \cdot M_o$", flux_units),
    panel(f.sdiv_a, raw"atmosphere $m_a \nabla \cdot M_a$", flux_units),
    panel(f.flux_o, raw"ocean flux form $-\nabla \cdot (M_o m_o)$", flux_units),
    panel(f.flux_a, raw"atmosphere flux form $-\nabla \cdot (M_a m_a)$", flux_units),
    panel(f.flux_o.+f.sdiv_o, raw"ocean sum $-\nabla \cdot (M_o m_o) + m_o \nabla \cdot M_o$", flux_units),
    panel(f.flux_a.+f.sdiv_a, raw"atmosphere sum $-\nabla \cdot (M_a m_a) + m_a \nabla \cdot M_a$", flux_units)]
fig3, axs3 = plot_panels(f.lon, f.lat, panels3; land=f.land, suptitle=figtitle);

## All 8 phases: zonal wind stress and surface-air MSE
Read for each phase only what these maps need (`taux`, `sst` for the land mask, `t2`, `q`, `nobs_taux`). All 8 panels of a figure share one color scale; the panel titles give the number of active days in each phase.

In [ ]:
dp = [read_group(file, findfirst(==(q), phases); vars=("taux", "sst", "t2", "q", "nobs_taux"), lonlim, latlim) for q in 1:8]
lon, lat, land = dp[1].lon, dp[1].lat, isnan.(dp[1].sst)
daytitle(q, x) = "phase $q ($(Int(maximum(x.nobs_taux))) days)"
taulim = maximum(x -> symlim(x.taux), dp) # N m^-2, shared by all phases
panels_tau = [panel(x.taux, daytitle(q, x), raw"N m$^{-2}$"; vmin=-taulim, vmax=taulim) for (q, x) in zip(1:8, dp)]
fig_tau, axs_tau = plot_panels(lon, lat, panels_tau; ncols=4, land, suptitle=raw"BSISO1 phase composites: zonal wind stress $\tau_x$");

In [ ]:
ma = [@. (c_pa*x.t2 + Lv*x.q) / 1e3 for x in dp] # m_a = c_pa T_2 + L_v q, kJ/kg
malim = extrema(m for x in ma for (m, l) in zip(x, land) if !l && isfinite(m)) # over ocean, all phases
panels_ma = [panel(m, daytitle(q, x), raw"kJ kg$^{-1}$"; cmap="viridis", vmin=malim[1], vmax=malim[2]) for (q, x, m) in zip(1:8, dp, ma)]
fig_ma, axs_ma = plot_panels(lon, lat, panels_ma; ncols=4, land, suptitle=raw"BSISO1 phase composites: surface-air MSE $m_a = c_{pa} T_2 + L_v q$");

## Save figures

In [ ]:
# fig1.savefig("bsiso_V_ma_mo_phase$p.png")
# fig_adv.savefig("bsiso_adv_sum_phase$p.png")
# fig3.savefig("bsiso_sdiv_flux_phase$p.png")
# fig_tau.savefig("bsiso_taux_phases.png")
# fig_ma.savefig("bsiso_ma_phases.png")